# MOGA-Phonons correlation heatmaps for dataframe000013–dataframe000016 with derived parameters

This notebook aggregates four independent MOGA-Phonons post-processing dataframes:

- `dataframe000013.pkl`
- `dataframe000014.pkl`
- `dataframe000015.pkl`
- `dataframe000016.pkl`

It keeps only the **top 5 overall** solutions for each `(dataset, mass, lattice parameter)` combination, ranked by `fitness_norm`, and generates publication-quality Pearson and Spearman correlation heatmaps.

The selection is **not** top 5 per generation. It is top 5 overall by `fitness_norm` for each independent dataframe and each `(m, a)` pair.

Two derived BvK combinations are added to the correlation analysis:

\[
\alpha_1 + 2\beta_1
\]

\[
\alpha_1 - \beta_1
\]


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional but useful for high-resolution notebook display.
%config InlineBackend.figure_format = "retina"

# -----------------------------------------------------------------------------
# User settings
# -----------------------------------------------------------------------------

# Update this path if your pickle files are somewhere else.
DATA_DIR = Path("./dataframes/")   # e.g., Path("/Users/jamunoz/Documents/GitHub/MOGA-Phonons/dataframes")

DATAFRAME_FILES = {
    "dataframe000013": DATA_DIR / "dataframe000013.pkl",
    "dataframe000014": DATA_DIR / "dataframe000014.pkl",
    "dataframe000015": DATA_DIR / "dataframe000015.pkl",
    "dataframe000016": DATA_DIR / "dataframe000016.pkl",
}

OUTPUT_DIR = Path("correlation_heatmaps_top5_overall_dataframe000013_000016_with_derived")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 5
RANK_BY = "fitness_norm"

# Figure settings.
FIGSIZE = (11.0, 9.0)
DPI = 300
SAVE_FORMATS = ("pdf", "png")


In [ ]:
def find_first_existing(candidates, columns):
    """Return the first column name from candidates that exists in columns."""
    for col in candidates:
        if col in columns:
            return col
    return None


def load_dataframe(path, dataset_label):
    """Load a pickle dataframe and add a dataset label."""
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Update DATA_DIR or DATAFRAME_FILES at the top of the notebook."
        )
    df = pd.read_pickle(path)
    df = df.copy()
    df["dataset"] = dataset_label
    return df


def rename_for_display(columns):
    """Make compact, LaTeX-like labels for heatmap axes."""
    label_map = {
        "mass": "$m$",
        "m": "$m$",
        "a_val": "$a_0$",
        "alat": "$a_0$",
        "a_latt": "$a_0$",
        "lattice_parameter": "$a_0$",
        "alpha0": r"$\alpha_0$",
        "alpha_0": r"$\alpha_0$",
        "alpha1": r"$\alpha_1$",
        "alpha_1": r"$\alpha_1$",
        "beta1": r"$\beta_1$",
        "beta_1": r"$\beta_1$",
        "alpha2": r"$\alpha_2$",
        "alpha_2": r"$\alpha_2$",
        "beta2": r"$\beta_2$",
        "beta_2": r"$\beta_2$",
        "alpha1_plus_2beta1": r"$\alpha_1 + 2\beta_1$",
        "alpha1_minus_beta1": r"$\alpha_1 - \beta_1$",
        "fitness1": r"$f_1$",
        "f1": r"$f_1$",
        "fitness2": r"$f_2$",
        "f2": r"$f_2$",
        "fitness3": r"$f_3$",
        "f3": r"$f_3$",
        "fitness_norm": r"$f_{\mathrm{norm}}$",
        "fnorm": r"$f_{\mathrm{norm}}$",
        "min_frequency": r"$\omega_{\min}$",
        "min_freq": r"$\omega_{\min}$",
        "max_frequency": r"$\omega_{\max}$",
        "max_freq": r"$\omega_{\max}$",
        "num_imaginary": r"$N_{\mathrm{imag}}$",
        "n_imaginary": r"$N_{\mathrm{imag}}$",
    }
    return [label_map.get(col, col) for col in columns]


def add_derived_bvk_parameters(df):
    """
    Add derived nearest-neighbor BvK combinations:
        alpha1_plus_2beta1 = alpha1 + 2 beta1
        alpha1_minus_beta1 = alpha1 - beta1

    The function accepts either alpha1/beta1 or alpha_1/beta_1 naming.
    """
    alpha1_col = find_first_existing(["alpha1", "alpha_1"], df.columns)
    beta1_col = find_first_existing(["beta1", "beta_1"], df.columns)

    if alpha1_col is None or beta1_col is None:
        raise ValueError(
            "Could not compute derived parameters. Expected alpha1/beta1 or alpha_1/beta_1 columns."
        )

    df = df.copy()
    df["alpha1_plus_2beta1"] = pd.to_numeric(df[alpha1_col], errors="coerce") + 2.0 * pd.to_numeric(df[beta1_col], errors="coerce")
    df["alpha1_minus_beta1"] = pd.to_numeric(df[alpha1_col], errors="coerce") - pd.to_numeric(df[beta1_col], errors="coerce")
    return df


In [ ]:
# -----------------------------------------------------------------------------
# Load and aggregate the four independent dataframes
# -----------------------------------------------------------------------------

frames = []
for label, path in DATAFRAME_FILES.items():
    tmp = load_dataframe(path, label)
    print(f"{label}: {tmp.shape[0]:,} rows, {tmp.shape[1]:,} columns")
    frames.append(tmp)

df_all = pd.concat(frames, ignore_index=True)
print(f"\nAggregated dataframe before filtering: {df_all.shape[0]:,} rows, {df_all.shape[1]:,} columns")
df_all.head()


In [ ]:
# -----------------------------------------------------------------------------
# Detect key column names
# -----------------------------------------------------------------------------

mass_col = find_first_existing(["mass", "m"], df_all.columns)
alat_col = find_first_existing(["a_val", "alat", "a_latt", "lattice_parameter"], df_all.columns)
rank_col = find_first_existing([RANK_BY, "fitness_norm", "fnorm"], df_all.columns)

if mass_col is None:
    raise ValueError("Could not find a mass column. Expected one of: mass, m")

if alat_col is None:
    raise ValueError("Could not find a lattice-parameter column. Expected one of: a_val, alat, a_latt, lattice_parameter")

if rank_col is None:
    raise ValueError("Could not find a ranking column. Expected fitness_norm or fnorm.")

print(f"Detected mass column: {mass_col}")
print(f"Detected lattice-parameter column: {alat_col}")
print(f"Detected ranking column: {rank_col}")


In [ ]:
# -----------------------------------------------------------------------------
# Keep only the top TOP_K overall per dataset, mass, and lattice parameter
# -----------------------------------------------------------------------------
#
# This is the key filtering step.
#
# It selects the top TOP_K rows after sorting by fitness_norm for each group:
#     dataset, mass, lattice parameter
#
# It does NOT group by generation, so the selected rows are the top TOP_K overall
# for each independent dataframe and each (m, a) pair.
# -----------------------------------------------------------------------------

df = (
    df_all.sort_values(rank_col, ascending=False)
    .groupby(["dataset", mass_col, alat_col], group_keys=False)
    .head(TOP_K)
    .copy()
)

df = add_derived_bvk_parameters(df)

print(f"Filtered dataframe after top-{TOP_K}-overall selection: {df.shape[0]:,} rows")
print("Added derived columns: alpha1_plus_2beta1, alpha1_minus_beta1")

summary = (
    df.groupby(["dataset", mass_col, alat_col])
    .size()
    .reset_index(name="n_selected")
)

print("\nSelection count summary:")
print(summary["n_selected"].describe())

df.head()


In [ ]:
# Optional sanity check: show the selected rows for one mass/lattice-parameter pair.

example_mass = sorted(df[mass_col].dropna().unique())[0]
example_alat = sorted(df[alat_col].dropna().unique())[0]

check = df[
    np.isclose(df[mass_col].astype(float), float(example_mass))
    & np.isclose(df[alat_col].astype(float), float(example_alat))
].sort_values(["dataset", rank_col], ascending=[True, False])

display_cols = [
    c for c in [
        "dataset", mass_col, alat_col, "generation", "solution_idx", "rank", rank_col,
        "alpha1_plus_2beta1", "alpha1_minus_beta1"
    ]
    if c in check.columns
]

check[display_cols].head(30)


In [ ]:
# -----------------------------------------------------------------------------
# Build the scalar dataframe used for correlations
# -----------------------------------------------------------------------------

preferred_groups = [
    ["mass", "m"],
    ["a_val", "alat", "a_latt", "lattice_parameter"],
    ["alpha0", "alpha_0"],
    ["alpha1", "alpha_1"],
    ["beta1", "beta_1"],
    ["alpha1_plus_2beta1"],
    ["alpha1_minus_beta1"],
    ["alpha2", "alpha_2"],
    ["beta2", "beta_2"],
    # Uncomment these if you want to include the individual and normalized fitness values.
    # ["fitness1", "f1"],
    # ["fitness2", "f2"],
    # ["fitness3", "f3"],
    # ["fitness_norm", "fnorm"],
    # ["min_frequency", "min_freq"],
    ["max_frequency", "max_freq"],
    # ["num_imaginary", "n_imaginary"],
]

correlation_columns = []
for group in preferred_groups:
    col = find_first_existing(group, df.columns)
    if col is not None:
        correlation_columns.append(col)

# Remove accidental duplicates while preserving order.
correlation_columns = list(dict.fromkeys(correlation_columns))

corr_df = df[correlation_columns].copy()

# Convert to numeric and drop rows with missing scalar values.
for col in corr_df.columns:
    corr_df[col] = pd.to_numeric(corr_df[col], errors="coerce")

before = corr_df.shape[0]
corr_df = corr_df.dropna(axis=0, how="any")
after = corr_df.shape[0]

print("Correlation columns:")
for col in correlation_columns:
    print(f"  {col}")

print(f"\nRows used for correlation: {after:,} / {before:,}")
corr_df.describe().T


In [ ]:
# -----------------------------------------------------------------------------
# Compute Pearson and Spearman correlation matrices
# -----------------------------------------------------------------------------

pearson_corr = corr_df.corr(method="pearson")
spearman_corr = corr_df.corr(method="spearman")

display_labels = rename_for_display(correlation_columns)

pearson_corr_display = pearson_corr.copy()
spearman_corr_display = spearman_corr.copy()

pearson_corr_display.index = display_labels
pearson_corr_display.columns = display_labels
spearman_corr_display.index = display_labels
spearman_corr_display.columns = display_labels

pearson_corr_display


In [ ]:
def plot_correlation_heatmap(corr, title, stem):
    """
    Plot and save a publication-quality correlation heatmap using matplotlib only.
    """
    fig, ax = plt.subplots(figsize=FIGSIZE)

    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="coolwarm", origin="upper")

    # Ticks and labels.
    ax.set_xticks(np.arange(corr.shape[1]))
    ax.set_yticks(np.arange(corr.shape[0]))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right", rotation_mode="anchor", fontsize=16)
    ax.set_yticklabels(corr.index, fontsize=16)

    # Cell annotations.
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            val = corr.values[i, j]
            text_color = "white" if abs(val) > 0.7 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=11, color=text_color)

    # Grid lines.
    ax.set_xticks(np.arange(-0.5, corr.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, corr.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=1.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Colorbar.
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=14)
    cbar.set_label("Correlation coefficient", fontsize=16)

    ax.set_title(title, fontsize=18, pad=18)
    fig.tight_layout()

    saved_paths = []
    for fmt in SAVE_FORMATS:
        out = OUTPUT_DIR / f"{stem}.{fmt}"
        fig.savefig(out, dpi=DPI, bbox_inches="tight")
        saved_paths.append(out)

    print("Saved:")
    for out in saved_paths:
        print(f"  {out}")

    return fig, ax, saved_paths


In [ ]:
fig, ax, saved_pearson = plot_correlation_heatmap(
    pearson_corr_display,
    title="Pearson correlation of aggregated top 5 solutions per run",
    stem="pearson_correlation_heatmap_top5_overall_with_derived",
)


In [ ]:
fig, ax, saved_spearman = plot_correlation_heatmap(
    spearman_corr_display,
    title="Spearman correlation of aggregated top 5 solutions per run",
    stem="spearman_correlation_heatmap_top5_overall_with_derived",
)


In [ ]:
# -----------------------------------------------------------------------------
# Save numerical correlation matrices as CSV files
# -----------------------------------------------------------------------------

pearson_csv = OUTPUT_DIR / "pearson_correlation_matrix_top5_overall_with_derived.csv"
spearman_csv = OUTPUT_DIR / "spearman_correlation_matrix_top5_overall_with_derived.csv"

pearson_corr.to_csv(pearson_csv)
spearman_corr.to_csv(spearman_csv)

print("Saved numerical matrices:")
print(f"  {pearson_csv}")
print(f"  {spearman_csv}")


## Notes for manuscript use

This notebook performs the correlation analysis on the top five overall solutions, ranked by \(f_{\mathrm{norm}}\), for each independent dataset and each \((m,a)\) pair. This selection emphasizes the best solutions produced by each optimization sweep rather than sampling the best individuals from every generation.

The derived quantities

\[
\alpha_1 + 2\beta_1
\]

and

\[
\alpha_1 - \beta_1
\]

are included alongside the original scalar parameters in the correlation matrix.

The output files are saved in:

```text
correlation_heatmaps_top5_overall_dataframe000013_000016_with_derived/
```

The main figure files are:

```text
pearson_correlation_heatmap_top5_overall_with_derived.pdf
spearman_correlation_heatmap_top5_overall_with_derived.pdf
```
